# Лабораторна 5. Завдання 4 (частина 2) — розпізнавання власних цифр збереженою CNN
Пункти 34–43. Потрібні: файл `cnn_mnist_final.keras` та 10 файлів `digits/0.png … digits/9.png`
(темна цифра від руки на білому фоні). Завантажте їх у Colab через панель Files.

## 36. Імпорт

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow import keras

## 37–38. Завантаження збереженої мережі та перекомпіляція

In [ ]:
model = keras.models.load_model("cnn_mnist_final.keras")
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

## 39–42. Завантаження малюнків, масив, нормалізація, інверсія, додатковий вимір, розпізнавання

In [ ]:
def prepare(path):
    img = Image.open(path).convert("L").resize((28, 28))    # відтінки сірого 28x28
    arr = np.array(img).astype("float32") / 255             # масив + нормалізація
    arr = 1.0 - arr                                         # інверсія: MNIST — біла цифра на чорному
    return arr.reshape(1, 28, 28, 1)                        # додаємо вимір пакета (batch) і канал

true, pred = [], []
fig, axes = plt.subplots(1, 10, figsize=(16, 2.4))
for d, a in zip(range(10), axes):
    x = prepare(f"digits/{d}.png")
    probs = model.predict(x, verbose=0)[0]
    p = int(probs.argmax())
    true.append(d); pred.append(p)
    a.imshow(x[0, :, :, 0], cmap="gray"); a.axis("off")
    a.set_title(f"{p} ({probs.max():.0%})\n{'ok' if p == d else 'ПОМИЛКА'}", fontsize=9)
plt.savefig("cnn_own_predictions.png", dpi=150, bbox_inches="tight"); plt.show()
print("Правильно:", sum(int(a == b) for a, b in zip(true, pred)), "із 10")

## 43. Висновки (заповнити числами за результатами запуску)
- Кращою виявилась найглибша CNN (Conv + BatchNorm + Dropout): точність на тесті __ %, 5-fold: __ ± __ %.
  Для порівняння: MLP — __ %, проста CNN — __ %.
- Висока точність зберігається і на валідації/тесті (різниця train/val мала), на 10 зображеннях датасету розпізнано __ із 10,
  на власних — __ із 10. На власних нижче через відмінності стилю письма, товщини лінії і фону.
- Проти перенавчання: Dropout, BatchNormalization, EarlyStopping за val_loss з відновленням найкращих ваг,
  ModelCheckpoint, k-fold крос-валідація, контроль кривих навчання (розбіжність train/val), нормалізація даних.